# whisper-fine-tuning — walkthrough — Whisper fine-tuning + emotion tokens

This notebook narrates the same happy path the [README](../../services/whisper-fine-tuning/README.md)'s Usage section describes,
calling the real project modules in sequence rather than reimplementing their logic. It is the
interactive counterpart to `make demo` / `make train` / `make prepare-data`.

**Before running:** `make install` (which runs `uv sync`), plus `jupyter`/`ipykernel` to actually
execute this notebook. Launch Jupyter from inside that same environment (`uv run jupyter notebook`,
or select `.venv`'s interpreter as the kernel) so the `!python ...` cells below resolve the right
interpreter.


In [ ]:
import os
from pathlib import Path

# nbconvert / Jupyter execute this notebook with cwd = this file's own directory
# (notebooks/) by default. Chdir to the repo root once, up front, so every relative
# path below (data/, datasets/, checkpoints/) matches the README's own repo-root-relative
# paths -- guarded so re-running this cell is a no-op.
if not Path("Makefile").exists():
    os.chdir(os.path.join("..", "..", "services", "whisper-fine-tuning"))
assert Path("Makefile").exists(), "expected to be at the repo root after chdir"

# No path manipulation needed beyond that: the flat modules at the repo root import
# directly, so `import config` and friends resolve from anywhere.

## Part 1 — single-example fine-tuning demo

`demo` loads Whisper-tiny, transcribes `data/audio.mp3` with the base
model, takes one Adam step toward a supplied `target_text`, then transcribes again. The cells below
call the same functions the demo module calls — nothing here is a reimplementation.


In [ ]:
import platform
import subprocess
from pathlib import Path

target_text = "My name is Rosh"
audio_path = "data/audio.mp3"

Path(audio_path).parent.mkdir(parents=True, exist_ok=True)
if not Path(audio_path).exists():
    if platform.system() == "Darwin":
        subprocess.run(["say", "-o", "clip.aiff", target_text], check=True)
        subprocess.run(["ffmpeg", "-y", "-i", "clip.aiff", audio_path], check=True)
        print(f"Synthesized {audio_path} from: {target_text!r}")
    else:
        raise FileNotFoundError(
            f"{audio_path} not found and `say` is macOS-only. Supply your own short spoken "
            "audio file at this path (and update target_text below to match what it says)."
        )
else:
    print(f"Using existing {audio_path} (delete it to re-synthesize).")

In [ ]:
from config import Config
from models import decode_audio, get_tokenizer, load_whisper_model
from training import fine_tune

config = Config()
model, device = load_whisper_model(config)
tokenizer = get_tokenizer()

base_output = decode_audio(model, device, audio_path)
print("Base model output:", base_output)

In [ ]:
model = fine_tune(
    model, tokenizer, device, audio_path, target_text, learning_rate=config.learning_rate
)
fine_tuned_output = decode_audio(model, device, audio_path)

print("Base model output:      ", base_output)
print("Fine-tuned model output:", fine_tuned_output)

One gradient step measurably shifts the transcription toward the correction, but a single step
isn't enough to fully converge — see Section 6 of the README for the documented loss numbers
(2.882 → 1.574) and a discussion of exactly what does and doesn't improve after one step.

The interactive record/compare/fine-tune loop (`ui/app.py`, `make ui` / `streamlit run ui/app.py`) does
the same thing through a browser UI with a live mic — not drivable from this notebook, see
Section 6's Figure 8 for a screenshot of it running.

## Part 2 — the emotion-aware data pipeline

Builds everything up to (but not including) training Whisper on emotion-labelled speech: fetch
data, organize it by emotion, split train/val, build CSV manifests, add `<emotion_*>` tokens to
the tokenizer, and run some EDA. See README Section 2.4 for the full picture.

### Download

The real entry point is `python -m data_prep.download_dataset`, but
its `main()` also calls `setup_crema_d()`, which triggers a multi-gigabyte git-lfs clone. That is an
expensive, un-asked-for operation for a walkthrough (the same reasoning `TODO.md` documents for why
CREMA-D was never fetched during the code-hygiene passes), so this cell calls `setup_emodb()`
directly — the same real function, without the CREMA-D side effect. It is guarded to skip
re-downloading if EmoDB is already on disk.


In [ ]:
from data_prep.download_dataset import setup_emodb

if Path("datasets/emodb/wav").exists():
    print("EmoDB already present on disk — skipping re-download.")
else:
    setup_emodb(config)

In [ ]:
!python -m data_prep.prepare_dataset

In [ ]:
!python -m data_prep.create_manifest

In [ ]:
!python -m add_emotion_tokens

In [ ]:
!python -m data_prep.explore_data

In [ ]:
from IPython.display import Image, display

display(Image(filename="data/dataset_analysis.png"))

## Wrap-up

- Full experimental results, real numbers, and honestly-documented limitations (CREMA-D never
  downloaded, no training run on the emotion manifests yet) are in the README's Section 6.
- The same stages above are also available as `make train`, `make demo`, `make serve`, `make ui`
  and `make prepare-data` (or their `docker compose` / `make docker-*` equivalents).
